# Replicate Published Results

Before trusting what a method returns, rerun the same code on data whose results have been published by others, and compare the numbers one by one.

This tutorial is based on step 5 of the workflow **Finance: volatility of daily changes** (Compose > Workflows > Search Library). Its first code is the GARCH-Type Models panel's **Replicate** Next Step (Next Steps › Compare, GARCH tab). It then finds out why ARCHModels.jl does not reproduce the benchmark to all its digits.

In [ ]:
Pollis.packages("ARCHModels", "Optim")

## Why published results

Rewriting the same algorithm or method in another language is not a check: both versions apply the same formulas to the same data, so they agree even when the formulas, a default or the data are wrong. A published table was computed by someone else, with other software, on the original data; matching it checks the data, the estimates and the options together.

## The benchmark

Fiorentini, Calzolari and Panattoni (1996) computed GARCH(1, 1) estimates with analytic derivatives on the daily Deutsche mark / British pound returns of Bollerslev and Ghysels (1996), 1974 days: normal errors, a constant mean, and the presample squared residual and variance both set to the mean squared residual (Brooks, Burke and Persand 2001, Section 2, equations 1-4). Brooks, Burke and Persand (2001, Section 2) quote the estimates to 6 significant figures: mu -0.00619041, omega 0.0107613, alpha 0.153134, beta 0.805974. Of the nine packages they tested, only TSP reproduced them with its default settings (Section 4).

## 1. ARCHModels against the benchmark

Each value is shown in full, rounded to the 6 significant figures published, and beside the published number.

In [ ]:
using ARCHModels, Printf

# Benchmark of Fiorentini, Calzolari and Panattoni (1996), quoted to 6 significant figures in Brooks, Burke and Persand (2001, Section 2):
# GARCH(1, 1), normal errors, constant mean, on the daily DM/pound returns of Bollerslev and Ghysels (1996), 1974 days
benchmark = fit(GARCH{1, 1}, BG96)  # BG96: the same returns, shipped with ARCHModels
published = [0.0107613, 0.805974, 0.153134, -0.00619041]  # omega, beta, alpha, mu: the order of coef(benchmark)
@printf("%-8s%16s%14s%14s\n", "", "Julia", "rounded", "published")
for (name, e, p) in zip(coefnames(benchmark), coef(benchmark), published)
	digits = something(findlast(n -> round(e, sigdigits=n) == round(p, sigdigits=n), 1:6), 0)  # significant figures in common
	@printf("%-8s%16.10f%14s%14s  %s\n", name, e, string(round(e, sigdigits=6)), string(p), digits == 0 ? "DIFFERENT" : "same to $digits significant figures")
end
# The gap comes from the start-up: ARCHModels sets the first day's variance to the sample variance, the benchmark
# starts the recursion from the mean squared residual (Brooks, Burke and Persand 2001, Section 2, equation 4).
# Learn More > Replicate Published Results: with the benchmark's start-up the same likelihood gives the published values to 5 or 6 significant figures.

In [ ]:
# Brooks, Burke and Persand (2001, Section 4, equation 5) score accuracy by the log relative error: its integer part is the number of digits that agree
lre(x, c) = -log10(abs(x - c) / abs(c))
for (name, e, p) in zip(coefnames(benchmark), coef(benchmark), [0.0107613, 0.805974, 0.153134, -0.00619041])
	@printf("%-8s LRE %.2f\n", name, lre(e, p))
end

## 2. Why: the start-up

The conditional likelihood needs a squared residual and a variance for the day before the first one. Brooks, Burke and Persand (2001, Section 2) warn that packages choose these by default, often without telling the user, and that "many published papers do not even discuss the initialisation of pre-sample values".

ARCHModels sets the first day's variance to the sample variance, fixed before the estimation. The benchmark starts the recursion from the mean squared residual (equation 4). To see whether this explains the gap, the cell below writes the likelihood out and maximises it under three start-ups:

- **ARCHModels' start-up:** should give ARCHModels' estimates;
- **equation 4, residuals of a first-pass regression** on a constant, fixed during the estimation;
- **equation 4, residuals at the current mu**, recomputed as mu changes.

This is a diagnosis, not a validation: it is our own rewrite of the likelihood, so it can only show where the difference comes from.

In [ ]:
using Optim, Statistics

# Conditional likelihood of GARCH(1, 1) with normal errors and a constant mean (Brooks, Burke and Persand 2001, equations 1-3),
# under three start-ups for the presample squared residual and variance
y = BG96; T = length(y)
function negloglik(theta, startup)
	mu, omega, alpha, beta = theta
	e = y .- mu
	h0 = startup == :archmodels ? var(y) :        # ARCHModels: the sample variance (divisor T - 1), used as the first day's variance
	     startup == :ols ? mean(abs2, y .- mean(y)) :  # equation 4 with the residuals of the first-pass regression, fixed
	     mean(abs2, e)                             # equation 4 with the residuals at the current mu
	s = 0.0; h = h0; e2 = h0
	for t in 1:T
		h = (startup == :archmodels && t == 1) ? h0 : omega + alpha * e2 + beta * h
		h <= 0 && return Inf
		s += 0.5 * (log(h) + e[t]^2 / h)
		e2 = e[t]^2
	end
	return s
end
fits = Dict(s => optimize(theta -> negloglik(theta, s), [-0.006, 0.01, 0.15, 0.8], BFGS(), Optim.Options(g_tol = 1e-12); autodiff = :forward) for s in (:archmodels, :ols, :current))
all(Optim.converged, values(fits)) || error("an optimisation did not converge")
own = Dict(s => Optim.minimizer(r) for (s, r) in fits)  # mu, omega, alpha, beta
published = [-0.00619041, 0.0107613, 0.153134, 0.805974]
package = coef(benchmark)[[4, 1, 3, 2]]  # ARCHModels' estimates in the same order
same(a, b) = something(findlast(n -> round(a, sigdigits=n) == round(b, sigdigits=n), 1:6), 0)  # significant figures in common
@printf("%-7s%14s%16s%16s%16s%13s  %s\n", "", "ARCHModels", "own: its start", "own: eq. 4 OLS", "own: eq. 4 mu", "published", "eq. 4 mu vs published")
for (i, name) in enumerate(["mu", "omega", "alpha", "beta"])
	@printf("%-7s%14.8f%16.8f%16.8f%16.8f%13s  same to %d significant figures\n", name, package[i], own[:archmodels][i], own[:ols][i], own[:current][i], string(published[i]), same(own[:current][i], published[i]))
end

## What the diagnosis shows

Read the table above:

- With ARCHModels' start-up, the rewritten likelihood gives ARCHModels' estimates: the optimiser and the likelihood formula are not the cause.
- With equation 4 and the residuals recomputed at the current mu, it gives the published estimates to 5 significant figures (mu, omega) or 6 (alpha, beta).
- Fixing the residuals at the first-pass regression is not enough, mostly for mu.

So ARCHModels is not wrong: it estimates a slightly different model, because a GARCH model estimated by conditional maximum likelihood is defined by its start-up as well. On these 1974 days the start-up moves the estimates in the third to sixth significant figure.

**For your own work:** state the start-up when you report GARCH estimates, and when you compare packages, compare them on the same start-up.

## References

- Bollerslev, T. and Ghysels, E. (1996). Periodic Autoregressive Conditional Heteroscedasticity. *Journal of Business & Economic Statistics* 14(2), 139-151. [doi:10.2307/1392425](https://doi.org/10.2307/1392425)
- Brooks, C., Burke, S. P. and Persand, G. (2001). Benchmarks and the Accuracy of GARCH Model Estimation. *International Journal of Forecasting* 17(1), 45-56. [doi:10.1016/S0169-2070(00)00070-4](https://doi.org/10.1016/S0169-2070(00)00070-4)
- Fiorentini, G., Calzolari, G. and Panattoni, L. (1996). Analytic Derivatives and the Computation of GARCH Estimates. *Journal of Applied Econometrics* 11(4), 399-417. [doi:10.1002/(SICI)1099-1255(199607)11:4<399::AID-JAE401>3.0.CO;2-R](https://doi.org/10.1002/(SICI)1099-1255(199607)11:4%3C399::AID-JAE401%3E3.0.CO;2-R)